# TMI isocenter: evaluate a trained model in Google Colab

This notebook loads a model trained with the **training notebook** (`tmi-isocenter-training.ipynb`) and evaluates it on the **test patients**, which the model never saw during training. For each patient it computes the error (MSE) and draws the predicted treatment fields next to the real ones.

### Before you start
1. You need a trained model. By default the notebook uses the **latest run saved to Google Drive** by Step 7 of the training notebook (`MyDrive/tmi-isocenter-results`). To use another model, see **Settings**.
2. A GPU is **not** needed: the default CPU runtime is fine.
3. **Run all cells:** menu *Runtime → Run all*.
3. The first cell **restarts the session** (Colab shows a *"session crashed"* or *"restarting"* message: this is expected). When it happens, do **Runtime → Run all** again. The first cell then skips the installation and the rest of the notebook runs.

⏱️ Expected time: ~ 5 min for setup, then ~ 1 min for the evaluation.

## Step 1: Install conda

The project requires **Python 3.11**, while Colab ships a newer Python. We install `conda` to create a separate Python 3.11 environment. **This cell restarts the session the first time it runs.**

In [ ]:
!pip install -q "condacolab<0.2"
import condacolab

condacolab.install()  # restarts the session the first time; does nothing afterwards

## Settings

In [ ]:
# Leave empty to use the latest run saved in MyDrive/tmi-isocenter-results.
# Otherwise, the path of a .ckpt file, e.g. one uploaded with the file browser on the left:
# CHECKPOINT_PATH = "/content/my_model.ckpt"
CHECKPOINT_PATH = ""

# "auto": guessed from the run folder name (e.g. "body_version_0"). Otherwise "body" or "arms":
# it must match the model type used for training.
MODEL = "auto"

N_PLOTS = 5  # number of test patients to display (the test set has 11 patients)

## Step 2: Find the trained model

In [ ]:
import glob
import os

RESULTS_DIR = "/content/drive/MyDrive/tmi-isocenter-results"

if CHECKPOINT_PATH:
    checkpoint = CHECKPOINT_PATH
    run_dir = None
else:
    from google.colab import drive

    drive.mount("/content/drive")
    runs = sorted(glob.glob(f"{RESULTS_DIR}/*_version_*"), key=os.path.getmtime)
    assert (
        runs
    ), f"No saved runs in {RESULTS_DIR}: run Step 7 of the training notebook first."
    run_dir = runs[-1]
    checkpoints = sorted(
        glob.glob(f"{run_dir}/checkpoints/*.ckpt"), key=os.path.getmtime
    )
    assert checkpoints, f"No checkpoint found in {run_dir}/checkpoints"
    checkpoint = checkpoints[-1]

assert os.path.exists(checkpoint), f"Checkpoint not found: {checkpoint}"

run_name = (
    os.path.basename(run_dir)
    if run_dir
    else os.path.splitext(os.path.basename(checkpoint))[0]
)
if MODEL == "auto":
    MODEL = run_name.split("_")[0]
    assert MODEL in (
        "body",
        "arms",
    ), f"Cannot guess the model type from {run_name!r}: set MODEL in the Settings cell."

print("Checkpoint:", checkpoint)
print("Model type:", MODEL)

## Step 3: Download the code and the data

In [ ]:
import os

REPO_DIR = "/content/tmi-isocenter"

if not os.path.exists(REPO_DIR):
    !git clone --branch daihs --depth 1 https://github.com/nlambriICH/tmi-isocenter.git {REPO_DIR}
else:
    # Update to the latest version of the code (your own edits are kept)
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!git log -1 --format="Code version: %h (%cd)" --date=short

In [ ]:
# Download the dataset shared by the instructor on Google Drive
try:
    import gdown
except ImportError:
    !pip install -q gdown
    import gdown

DATA_FOLDER_ID = "1QRU3aQf_wFEVIWrHMJkHisoK8FxRQQUo"
DATA_DIR = f"{REPO_DIR}/data/5_355"

if not os.path.exists(f"{DATA_DIR}/raw/ptv_imgs2D.npz"):
    gdown.download_folder(
        id=DATA_FOLDER_ID, output=DATA_DIR, quiet=True, use_cookies=False
    )

expected_files = ["patient_info.csv"] + [
    f"raw/{name}"
    for name in [
        "ptv_imgs2D.npz",
        "ptv_masks2D.npz",
        "brain_masks2D.npz",
        "lungs_masks2D.npz",
        "liver_masks2D.npz",
        "intestine_masks2D.npz",
        "bladder_masks2D.npz",
        "isocenters_pix.npy",
        "jaws_X_pix.npy",
        "jaws_Y_pix.npy",
        "angles.npy",
    ]
]
missing = [f for f in expected_files if not os.path.exists(f"{DATA_DIR}/{f}")]
assert (
    not missing
), f"Missing data files: {missing}. Re-run this cell or ask the instructor."
print("Dataset ready ✅")

In [ ]:
# Create the configuration file from the template, using the settings above
import json

config_path = f"{REPO_DIR}/src/config/constants.json"
with open(f"{REPO_DIR}/src/config/constants.example.json") as f:
    config = json.load(f)

config["model"] = MODEL
config[
    "coll_5_355"
] = True  # the shared dataset uses the 5°/355° pelvis collimator configuration
config["num_workers"] = 2  # Colab provides 2 CPU cores

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)
print(f"Configuration saved: model = {MODEL!r}")

## Step 4: Create the Python 3.11 environment and install the dependencies

All project commands run inside this environment with `conda run -n tmi_env ...`. Takes 3-5 min, only the first time.

In [ ]:
ENV_READY_FLAG = "/content/.tmi_env_ready"

if not os.path.exists(ENV_READY_FLAG):
    !conda create -y -q -n tmi_env python=3.11
    !conda run -n tmi_env pip install -q -r requirements.txt
    !conda run -n tmi_env pip install -q -e .
    # PyTorch Lightning 2.0 needs pkg_resources, removed from recent setuptools
    !conda run -n tmi_env pip install -q "setuptools<70"
    open(ENV_READY_FLAG, "w").close()
else:
    print("Environment already installed.")

!conda run -n tmi_env python -c "import torch; print('Python OK | PyTorch', torch.__version__, '| GPU available:', torch.cuda.is_available())"

## Step 5: Pre-process the data

Combines the masks into 3-channel 512×512 images and transforms the labels accordingly (output in `data/5_355/interim`).

In [ ]:
if not os.path.exists(f"{DATA_DIR}/interim/masks2D.npy"):
    !conda run -n tmi_env python -m src.data.processing
print("Pre-processing done ✅")

## Step 6: Evaluate the model on the test patients

For each test patient, the model predicts the field geometry. The script computes the weighted mean squared error (MSE) between prediction and ground truth (the same loss used in training, on normalized coordinates: lower is better), and saves a plot of the predicted and of the real fields.

In [ ]:
EVAL_DIR = f"/content/evaluation/{run_name}"

!conda run -n tmi_env --no-capture-output python -u -m src.evaluate_cnn --ckpt "{checkpoint}" --output-dir "{EVAL_DIR}"

## Step 7: Look at the results

The table lists the error of each test patient. Below it, each row compares the **predicted** fields (left) with the fields planned by the medical physicist (**ground truth**, right).

In [ ]:
import base64
import csv

from IPython.display import HTML, display

with open(f"{EVAL_DIR}/test_metrics.csv") as f:
    metrics = sorted(csv.DictReader(f), key=lambda r: float(r["mse"]))

rows = "".join(
    f"<tr><td>{r['patient_id']}</td><td>{r['patient_idx']}</td><td>{float(r['mse']):.6f}</td></tr>"
    for r in metrics
)
mean_mse = sum(float(r["mse"]) for r in metrics) / len(metrics)
display(
    HTML(
        "<table><tr><th>Patient</th><th>Index</th><th>MSE</th></tr>"
        f"{rows}<tr><th>Mean</th><th></th><th>{mean_mse:.6f}</th></tr></table>"
    )
)


def img_tag(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return f'<img src="data:image/png;base64,{data}" style="width:100%">'


# Best patients first
for r in metrics[:N_PLOTS]:
    name = f"test_{r['patient_idx']}.png"
    display(
        HTML(
            f"<h4>Patient {r['patient_id']}: MSE = {float(r['mse']):.6f}</h4>"
            "<table style='width:100%'><tr><th>Predicted</th><th>Ground truth</th></tr><tr>"
            f"<td style='width:50%'>{img_tag(f'{EVAL_DIR}/img/test/predicted/{name}')}</td>"
            f"<td style='width:50%'>{img_tag(f'{EVAL_DIR}/img/test/ground_truth/{name}')}</td>"
            "</tr></table>"
        )
    )

## Step 8: Save the evaluation to Google Drive (optional)

Copies the plots and the metrics table into the run folder on Drive (`.../<run>/evaluation`), or into `MyDrive/tmi-isocenter-results/evaluation_<name>` for a checkpoint that does not come from Drive.

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
destination = (
    f"{run_dir}/evaluation" if run_dir else f"{RESULTS_DIR}/evaluation_{run_name}"
)
shutil.copytree(EVAL_DIR, destination, dirs_exist_ok=True)
print("Evaluation saved to", destination)

## What to try next

- Train models with different settings or code changes, then compare their mean test MSE here (set `CHECKPOINT_PATH` to evaluate an older run).
- Look at the patients with the highest error: what makes them difficult? (body size, arms position, organ masks, ...)